# Day 2, Session 3. From Text to Topics

Today you turn raw manifesto sentences into features, then into topics. First a document-term
matrix and an LDA topic model, both plain Python on the CPU. Then the Structural Topic Model,
which lets a covariate like left versus right shape what gets talked about.

## Install the R packages first

The Structural Topic Model at the end of this notebook runs in R. Run the next two cells as soon
as you open the notebook. R compiles the packages for several minutes, and you can follow the
session while it works. Move on once the second cell prints three package versions.

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
# stm needs tm and SnowballC for textProcessor, and a fresh Colab R lacks them
install.packages(c('stm', 'tm', 'SnowballC'), repos='https://cloud.r-project.org', quiet=TRUE)
for (p in c('stm', 'tm', 'SnowballC')) cat(p, as.character(packageVersion(p)), ' ')
cat('\n')

## Load the workshop helpers

One line fetches the helper functions we use across the workshop. Open
`workshop_utils.py` in the file browser on the left if you want to read them.

In [ ]:
!wget -q -O workshop_utils.py https://raw.githubusercontent.com/jacqpark/instats-python-workshop/main/workshop_utils.py
from workshop_utils import pull_manifesto, collapse_rile, manifesto_stopwords, LEFT_CODES, RIGHT_CODES
print('helpers loaded')

## Pull the corpus with your key

This uses the Manifesto key you stored in Colab secrets at the end of Day 1 and returns a table of quasi-sentences with
their CMP category. The data is pulled, never redistributed.

In [ ]:
from google.colab import userdata
import pandas as pd

# Your key, your copy. This takes about half a minute.
df = pull_manifesto(userdata.get('MANIFESTO_KEY'))
df[['text', 'cmp_code', 'partyname', 'countryname', 'year']].head()

## Collapse the categories to left and right

The Manifesto scheme has dozens of categories, each one a numeric code. We fold the standard
left and right sets into a single left-right label and drop the rest.

In [ ]:
# Every quasi-sentence carries a numeric Manifesto code. The standard
# left-right scale (RILE) uses 13 codes per side, and pull_manifesto has
# already folded them into the 'rile' column.
for c, name in list(LEFT_CODES.items())[:3]:
    print('left  ', c, name)
for c, name in list(RIGHT_CODES.items())[:3]:
    print('right ', c, name)

print()
print(collapse_rile(504), 'is what code 504 maps to')
print(collapse_rile('NA'), 'is what an uncoded sentence maps to')
print()
print(df['rile'].value_counts(dropna=False))

# Keep the labelled sentences. These are what the topic model reads and what
# Day 3 trains on.
df = df[df['rile'].notna()].reset_index(drop=True)
print('\nworking corpus', df.shape)

## From text to features

A document-term matrix counts which words appear in which sentence. `CountVectorizer` builds it.
`TfidfVectorizer` downweights common words. Keep this TF-IDF, it is the baseline your Day 3
fine-tune has to beat.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Country and party names would otherwise win topics of their own, so we add
# them to the English stopword list. manifesto_stopwords reads them off the
# corpus, so it follows the data rather than a list someone has to maintain.
STOP = manifesto_stopwords(df)
print(len(STOP), 'stopwords, including', [w for w in STOP if w in ('labour', 'gael', 'zealand')])

count_vec = CountVectorizer(stop_words=STOP, min_df=10, max_df=0.4, token_pattern=r'[a-zA-Z]{3,}')
X_counts = count_vec.fit_transform(df['text'])
print('document-term matrix', X_counts.shape)

tfidf_vec = TfidfVectorizer(stop_words=STOP, min_df=10, max_df=0.4, token_pattern=r'[a-zA-Z]{3,}')
X_tfidf = tfidf_vec.fit_transform(df['text'])
print('tf-idf matrix (Day 3 baseline)', X_tfidf.shape)

## Fit an LDA topic model

LDA reads the document-term matrix and finds groups of words that travel together, the topics.
Each topic is a distribution over words. Fit it, then read the top words per topic.

In [ ]:
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation

K = 10
lda = LatentDirichletAllocation(n_components=K, learning_method='batch', max_iter=20,
                                random_state=42, n_jobs=-1)
doc_topic = lda.fit_transform(X_counts)
vocab = np.array(count_vec.get_feature_names_out())
for k, comp in enumerate(lda.components_):
    top = vocab[np.argsort(comp)[::-1][:8]]
    print(f'Topic {k:2d}: ' + ' '.join(top))

## Do the topics line up with left and right

Assign each sentence its strongest topic, then see how left and right sentences spread across
topics. Some topics lean, but a topic is a policy area, not a left-right score. That gap is why
Day 3 trains a model on the labels directly.

In [ ]:
df['topic'] = doc_topic.argmax(axis=1)
ct = pd.crosstab(df['topic'], df['rile'], normalize='index').mul(100).round(0)
print(ct.to_string())

## Meeting STM, topics with a covariate

LDA ignores who wrote a sentence. The Structural Topic Model lets a covariate shape topic
prevalence. Here the covariate is left versus right. In your own work it would be party or year,
which the Manifesto pull also carries. We run the genuine R `stm` from Python through rpy2,
which leans on the R you already know. A pure-Python cousin is tomotopy's DMR.

The R packages install once at the start of the session. This cell fits STM and reads the
left-versus-right effect on each topic.

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

# The covariate is the election year, real document metadata rather than the
# label we are trying to predict. Party works the same way, swap 'year' below.
sample = (df.sample(min(3000, len(df)), random_state=42)[['text', 'year', 'partyabbrev']]
            .reset_index(drop=True))
sample['year'] = sample['year'].astype(int)
with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv['meta'] = ro.conversion.get_conversion().py2rpy(sample)

ro.r('''
suppressMessages(library(stm))
proc <- textProcessor(as.character(meta$text), metadata=meta, verbose=FALSE)
out  <- prepDocuments(proc$documents, proc$vocab, proc$meta, lower.thresh=5, verbose=FALSE)
fit  <- stm(out$documents, out$vocab, K=10, prevalence=~s(year), data=out$meta,
            init.type='Spectral', max.em.its=40, seed=42, verbose=FALSE)
eff  <- estimateEffect(1:10 ~ s(year), fit, metadata=out$meta)
lab  <- labelTopics(fit, n=6)
for (k in 1:10) cat(sprintf('Topic %2d  |  %s\n', k, paste(lab$frex[k,], collapse=' ')))
cat('\nHow topic prevalence moves with the election year:\n')
print(summary(eff, topics=1:3))
''')

## What you built

You turned text into a document-term matrix, found policy topics with LDA, and used STM to see
how those topics split left and right. Tomorrow you meet the encoder that reads word order and
context, which a bag of words cannot.